# Chapter 4 Lab — Simulating Multi-Scale Competency Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/liquid-books/basal-cognition/blob/main/notebooks/ch04-lab-mca-sim.ipynb)

**Basal Cognition · Dr. Ernesto Lee**

---

## What you will build

A simple two-level simulation:

- A **top-level agent** (the "organism") holds a target value and broadcasts an error signal.
- A set of **cell agents** each read the error signal and push a value toward the target.
- The organism never tells any cell *what to do* — it only broadcasts *how far off* the current state is.
- You will damage one cell agent mid-run and observe whether the others compensate.

This is the newt tubule experiment in code. No new instructions issued. Same target reached.

**Estimated time:** 30–45 minutes

**No prior Python experience needed.** Every block has a comment explaining what it does.

In [ ]:
# Install dependencies (only needed if running outside Colab)
# In Colab, matplotlib and numpy are already available.
%pip install -q matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Set a random seed so results are reproducible
random.seed(42)
np.random.seed(42)

print('Libraries loaded. Ready to build.')

## Part 1 — Define the agents

The **Organism** has a setpoint. It measures the current collective state and broadcasts the error.

Each **Cell** reads the error signal and adjusts its own output. No cell knows the target directly — it only knows the error.

In [ ]:
class Organism:
    """Top-level agent. Holds the target. Broadcasts error signal. Does NOT tell cells what to do."""

    def __init__(self, target: float):
        self.target = target  # The setpoint — what 'done' looks like

    def error_signal(self, current_state: float) -> float:
        """Returns how far the current state is from the target. Positive = need to go up."""
        return self.target - current_state


class Cell:
    """Subordinate agent. Reads error signal. Adjusts its own output. Does not know the target."""

    def __init__(self, cell_id: int, sensitivity: float = 0.1, active: bool = True):
        self.cell_id = cell_id
        self.output = random.uniform(0, 5)  # Start with random output
        self.sensitivity = sensitivity       # How strongly it responds to error
        self.active = active                 # Damage flag — False = cell is disabled

    def step(self, error: float):
        """Adjust output based on the error signal. If damaged, do nothing."""
        if not self.active:
            return  # Damaged cell contributes nothing
        # Move toward reducing the error, with a small random jitter (biological noise)
        self.output += self.sensitivity * error + random.gauss(0, 0.05)
        self.output = max(0, self.output)  # Output cannot go negative


def collective_state(cells):
    """The organism reads the sum of all cell outputs as the current state."""
    return sum(c.output for c in cells)


print('Agent classes defined.')

## Part 2 — Run the simulation

We run 100 timesteps with all cells active. Then we damage one cell and run 100 more. Watch whether the remaining cells compensate.

In [ ]:
# --- Configuration ---
TARGET = 50.0        # The organism's setpoint
NUM_CELLS = 10       # How many cell agents
STEPS_BEFORE = 100   # Steps before damage
STEPS_AFTER = 100    # Steps after damage
DAMAGE_CELL_ID = 3   # Which cell we will disable

# --- Initialize ---
organism = Organism(target=TARGET)
cells = [Cell(cell_id=i, sensitivity=0.08) for i in range(NUM_CELLS)]

history = []  # Track collective state over time
damage_step = STEPS_BEFORE

# --- Run phase 1: all cells healthy ---
for step in range(STEPS_BEFORE):
    state = collective_state(cells)
    history.append(state)
    error = organism.error_signal(state)
    for cell in cells:
        cell.step(error)

# --- Damage one cell ---
print(f'Damaging cell {DAMAGE_CELL_ID} at step {STEPS_BEFORE}...')
cells[DAMAGE_CELL_ID].active = False
cells[DAMAGE_CELL_ID].output = 0  # Its contribution drops to zero

# --- Run phase 2: damaged ---
for step in range(STEPS_AFTER):
    state = collective_state(cells)
    history.append(state)
    error = organism.error_signal(state)
    for cell in cells:
        cell.step(error)

print(f'Simulation complete. {len(history)} total steps recorded.')

## Part 3 — Plot the results

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

steps = list(range(len(history)))
ax.plot(steps, history, color='steelblue', linewidth=2, label='Collective state')
ax.axhline(TARGET, color='lime', linestyle='--', linewidth=1.5, label=f'Target (setpoint = {TARGET})')
ax.axvline(damage_step, color='tomato', linestyle=':', linewidth=2, label=f'Cell {DAMAGE_CELL_ID} damaged')

ax.set_xlabel('Timestep', fontsize=12)
ax.set_ylabel('Collective state (sum of cell outputs)', fontsize=12)
ax.set_title('MCA Simulation: Does the system recover from cell damage?', fontsize=14)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

final_state = history[-1]
print(f'Final state: {final_state:.2f} | Target: {TARGET} | Error: {abs(final_state - TARGET):.2f}')

## Part 4 — Experiments to try

Each experiment tests one idea from the chapter. Run each block and observe what changes.

### Experiment A — Damage more cells
What happens when you disable 3 cells instead of 1? 5 cells? At what point does the system stop recovering?

In [ ]:
# TODO: Modify DAMAGE_CELL_ID to be a list [1, 3, 5]
# Then rerun the simulation loop above and compare plots.
# Hint: loop over the list and set each cell's active = False before phase 2.

# YOUR CODE HERE
raise NotImplementedError('Implement Experiment A — damage multiple cells')

### Experiment B — Cut the error signal

What happens if the organism stops broadcasting the error signal (simulating a broken communication channel between levels)? Do the cells still converge?

In [ ]:
# TODO: After step 50, set error = 0 for all remaining steps regardless of actual state.
# Observe whether the system drifts from target without the error signal.

# YOUR CODE HERE
raise NotImplementedError('Implement Experiment B — cut the error signal at step 50')

### Experiment C — Change the target mid-run

At step 100, change the organism's target from 50 to 80. Does the system adapt? How quickly?

In [ ]:
# TODO: At step 100, set organism.target = 80.
# Watch whether the collective state shifts toward the new target.
# This is the analog of a bioelectric target-morphology rewrite in Levin's lab experiments.

# YOUR CODE HERE
raise NotImplementedError('Implement Experiment C — change target mid-run')

## Deliverable

Write a short paragraph (5–8 sentences) answering these questions:

1. Did the system recover from single-cell damage? How many steps did recovery take?
2. What happened when you cut the error signal? What does that tell you about the role of the shared channel?
3. Connect one observation from your simulation to a specific example from Chapter 4. Use the vocabulary: setpoint, error signal, outcome specification, de-integration.

Paste your paragraph in the cell below.

**Your answer here:**

*(Double-click this cell and replace this text with your paragraph.)*